# Intermittent Demand Forecasting

This notebook covers forecasting sparse, irregular demand data using classical heuristics (Croston, TSB, SBA, ADIDA, IMAPA), hierarchical Bayesian formulations, and hurdle-based ensembles.

## Downloading the Data

We use the UCI Online Retail dataset, which contains transaction records with many zero-demand periods per SKU.

In [ ]:
import requests
import zipfile
import io
import os

if not os.path.exists("Online Retail.xlsx"):
    url = "https://archive.ics.uci.edu/static/public/352/online+retail.zip"
    r = requests.get(url)
    r.raise_for_status()
    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        z.extractall(".")
    print("Downloaded and extracted.")
else:
    print("Data already present.")

## Preparing Intermittent Panel Data

We aggregate daily sales per SKU to create a panel with many zeros — the hallmark of intermittent demand.

In [ ]:
import pandas as pd
import numpy as np

df_raw = pd.read_excel("Online Retail.xlsx", sheet_name=0)
df_raw["InvoiceDate"] = pd.to_datetime(df_raw["InvoiceDate"])

# Remove cancellations and invalid rows
df_raw = df_raw[~df_raw["InvoiceNo"].astype(str).str.startswith("C")]
df_raw = df_raw[(df_raw["Quantity"] > 0) & (df_raw["UnitPrice"] > 0)].copy()

df_raw["Date"] = df_raw["InvoiceDate"].dt.floor("d")

# Daily quantity per SKU
daily = (
    df_raw.groupby(["StockCode", "Date"])["Quantity"]
    .sum()
    .reset_index()
    .rename(columns={"StockCode": "unique_id", "Date": "ds", "Quantity": "y"})
)

print(f"Raw panel: {daily.shape[0]} rows, {daily['unique_id'].nunique()} SKUs")

In [ ]:
# Fill missing dates with zeros to expose the intermittent structure
date_range = pd.date_range(daily["ds"].min(), daily["ds"].max(), freq="D")

# Select SKUs with enough history but high intermittency
sku_stats = daily.groupby("unique_id").agg(
    n_days=("ds", "nunique"),
    total_qty=("y", "sum"),
).reset_index()

n_calendar_days = len(date_range)
sku_stats["demand_fraction"] = sku_stats["n_days"] / n_calendar_days

# Intermittent: present on < 30% of days, but with at least 10 demand events
intermittent_skus = sku_stats[
    (sku_stats["demand_fraction"] < 0.30) & (sku_stats["n_days"] >= 10)
]["unique_id"].tolist()

print(f"Found {len(intermittent_skus)} intermittent SKUs")

# Build full panel with zero-fill for a manageable subset
subset_skus = intermittent_skus[:200]
daily_sub = daily[daily["unique_id"].isin(subset_skus)].copy()

idx = pd.MultiIndex.from_product(
    [subset_skus, date_range], names=["unique_id", "ds"]
)
panel = (
    daily_sub.set_index(["unique_id", "ds"])
    .reindex(idx, fill_value=0)
    .reset_index()
)

print(f"Full panel: {panel.shape[0]} rows")
print(f"Zero fraction: {(panel['y'] == 0).mean():.2%}")

## Classical Intermittent Heuristics with StatsForecast

We compare Croston's method, SBA, TSB (Teunter-Syntetos-Babai), ADIDA, and IMAPA using the StatsForecast library.

In [ ]:
from statsforecast import StatsForecast
from statsforecast.models import ADIDA, CrostonClassic, IMAPA, TSB

models = [
    CrostonClassic(),
    ADIDA(),
    IMAPA(),
    TSB(alpha_d=0.2, alpha_p=0.2),
]

sf = StatsForecast(
    models=models,
    freq="D",
    n_jobs=-1,
)

# Train/test split: hold out last 28 days
cutoff = panel["ds"].max() - pd.Timedelta(days=28)
train_df = panel[panel["ds"] <= cutoff].copy()
test_df = panel[panel["ds"] > cutoff].copy()

print(f"Train: {train_df.shape[0]} rows, Test: {test_df.shape[0]} rows")

In [ ]:
sf.fit(train_df)
forecast_df = sf.predict(h=28)
forecast_df.head(10)

## Evaluation with Sparse-Appropriate Metrics

Standard MAPE fails on zero-heavy data. We use MAE and RMSSE instead.

In [ ]:
merged = test_df.merge(forecast_df.reset_index(), on=["unique_id", "ds"], how="inner")

model_names = ["CrostonClassic", "ADIDA", "IMAPA", "TSB"]
results = []

for model_name in model_names:
    errors = merged["y"] - merged[model_name]
    mae = errors.abs().mean()
    rmse = np.sqrt((errors ** 2).mean())
    me = errors.mean()
    results.append({"Model": model_name, "ME": me, "MAE": mae, "RMSE": rmse})

results_df = pd.DataFrame(results).set_index("Model")
results_df.round(4)

## Hurdle Model: Two-Stage Forecasting

A hurdle model decomposes the forecast into two stages:
1. A classifier predicts whether any demand occurs
2. A regressor predicts the magnitude, conditioned on occurrence

This prevents zero-inflation from biasing magnitude estimates.

In [ ]:
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor


class HurdleModel:
    """Two-stage hurdle model for intermittent demand forecasting."""

    def __init__(self):
        self.classifier = RandomForestClassifier(random_state=42)
        self.regressor = RandomForestRegressor(random_state=42)

    def fit(self, X, y):
        """Train the classifier on all data and regressor on non-zeros."""
        y_occurrence = (y > 0).astype(int)
        self.classifier.fit(X, y_occurrence)

        positive_idx = y > 0
        if np.any(positive_idx):
            self.regressor.fit(X[positive_idx], y[positive_idx])

        return self

    def predict(self, X):
        """Multiply occurrence probability by expected magnitude."""
        p_occurrence = self.classifier.predict_proba(X)[:, 1]
        expected_magnitude = self.regressor.predict(X)
        return p_occurrence * expected_magnitude

In [ ]:
# Prepare features for the hurdle model using a single representative SKU
sample_sku = subset_skus[0]
sku_panel = panel[panel["unique_id"] == sample_sku].copy().sort_values("ds")

# Create lag features
for lag in [1, 7, 14]:
    sku_panel[f"lag_{lag}"] = sku_panel["y"].shift(lag)
sku_panel["rolling_mean_7"] = sku_panel["y"].shift(1).rolling(7).mean()
sku_panel["rolling_nonzero_7"] = (
    (sku_panel["y"] > 0).shift(1).rolling(7).sum()
)
sku_panel["day_of_week"] = sku_panel["ds"].dt.dayofweek

sku_panel = sku_panel.dropna()

feature_cols = ["lag_1", "lag_7", "lag_14", "rolling_mean_7", "rolling_nonzero_7", "day_of_week"]

# Split
sku_train = sku_panel[sku_panel["ds"] <= cutoff]
sku_test = sku_panel[sku_panel["ds"] > cutoff]

X_train = sku_train[feature_cols].values
y_train = sku_train["y"].values
X_test = sku_test[feature_cols].values
y_test = sku_test["y"].values

hurdle = HurdleModel()
hurdle.fit(X_train, y_train)
hurdle_preds = hurdle.predict(X_test)

hurdle_mae = np.abs(y_test - hurdle_preds).mean()
print(f"Hurdle MAE for SKU {sample_sku}: {hurdle_mae:.4f}")

## Visualizing Intermittent vs Continuous Demand

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=False)

# Panel A: continuous demand (pick a high-frequency SKU)
cont_skus = sku_stats[
    (sku_stats["demand_fraction"] > 0.7) & (sku_stats["n_days"] >= 100)
]["unique_id"].iloc[:1].values

if len(cont_skus) > 0:
    cont_sku = cont_skus[0]
    cont_data = daily[daily["unique_id"] == cont_sku].sort_values("ds").tail(90)
    axes[0].plot(cont_data["ds"], cont_data["y"], color="steelblue")
    axes[0].set_title(f"Panel A: Continuous Demand (SKU {cont_sku})")
    axes[0].set_ylabel("Quantity")

# Panel B: intermittent demand
inter_data = panel[panel["unique_id"] == sample_sku].sort_values("ds").tail(90)
axes[1].bar(inter_data["ds"], inter_data["y"], color="coral", width=1)
axes[1].set_title(f"Panel B: Intermittent Demand (SKU {sample_sku})")
axes[1].set_ylabel("Quantity")
axes[1].set_xlabel("Date")

plt.tight_layout()
plt.show()

In [ ]:
# Visualize the hurdle model predictions vs actuals
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(sku_test["ds"], y_test, color="steelblue", alpha=0.7, label="Actual", width=1)
ax.plot(sku_test["ds"], hurdle_preds, color="coral", marker="o", markersize=4, label="Hurdle Prediction")
ax.set_title(f"Hurdle Model Forecast for SKU {sample_sku}")
ax.set_ylabel("Quantity")
ax.legend()
plt.tight_layout()
plt.show()